# 07 Validation: BEM against the analytical solution for a sphere

| | |
|---|---|
| **Purpose** | Show that OptimUS converges to the exact answer as the mesh is refined. This is the evidence that the other notebooks can be trusted. |
| **Inputs** | The problem of `01_basic_model`: 1 kHz plane wave along (1, 1, 0), bone sphere of radius 1 m in water. Element sizes 1.0, 0.75, 0.5, 0.25 m (CI mode) plus 0.1 m (full mode). |
| **Outputs** | Convergence plot of relative error against element size (`outputs/07_validation_1.png`); side-by-side fields for the coarsest and finest mesh (`outputs/07_validation_2.png`). |
| **Checks** | Every solve converged; the error decreases with every refinement; the finest mesh is under 2% (CI) or 1% (full) in the Frobenius norm; observed convergence order at least 0.8. |
| **Runtime** | About 4.5 min full (the 0.1 m mesh with 1605 vertices takes 2.5 min of it), 30 s CI mode (Docker image, 4 CPUs, 9 Nov 2026). |
| **Adapted from** | `notebooks/Validation.ipynb` (upstream OptimUS). |

## Physics notes

For a penetrable sphere of radius $R$ hit by a plane wave $e^{i k_+ \mathbf{x} \cdot \mathbf{d}}$,
the scattered field outside is a sum of spherical waves

$$ u^s(\mathbf{x}) = \sum_{n=0}^{\infty} a_n \, h_n^{(1)}(k_+ r) \, P_n(\hat{\mathbf{x}} \cdot \mathbf{d}) $$

and the field inside is $\sum_n b_n j_n(k_- r) P_n(\hat{\mathbf{x}} \cdot \mathbf{d})$. The coefficients
$a_n, b_n$ follow from continuity of pressure and of $\frac{1}{\rho}\partial_r p$ at $r = R$. The full
derivation is in the upstream `notebooks/Validation.ipynb`. OptimUS implements it as
`optimus.model.acoustics.Analytical`; 100 terms is far more than needed at $k_+ R \approx 4.2$.

The error measures are the relative Frobenius norm $\|p_{BEM} - p_{exact}\|_F / \|p_{exact}\|_F$ and the
relative maximum norm, both over the real part of the total field on a 2D grid.

> **To expand (Yash):** derive $a_n$ and $b_n$; explain the expected convergence order for piecewise linear elements and compare with the measured order printed below.

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "07_validation"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. Parameters

In [ ]:
frequency = 1000.0
direction = (1, 1, 0)
element_sizes = [1.0, 0.75, 0.5, 0.25] if CI_MODE else [1.0, 0.75, 0.5, 0.25, 0.1]
resolution = (100, 100) if CI_MODE else (250, 250)
bounding_box = [-3, 5, -3, 4]
plane_axes = (0, 1)
finest_error_limit = 0.02 if CI_MODE else 0.01

In [ ]:
source = optimus.source.create_planewave(frequency, direction=direction)
material_ext = optimus.material.load_material("water")
material_int = optimus.material.load_material("bone (cortical)")

## 2. Solve both models for each element size

In [ ]:
def plane_field(model):
    plane = optimus.postprocess.VisualisePlane(model)
    plane.create_computational_grid(resolution=resolution, bounding_box=bounding_box,
                                    plane_axes=plane_axes)
    plane.compute_fields()
    return np.real(plane.total_field_imshow)

results = []
for element_size in element_sizes:
    t0 = time.time()
    geometry = optimus.geometry.shapes.Sphere(element_size=element_size)
    analytical = optimus.model.acoustics.Analytical(source, geometry, material_ext, material_int)
    analytical.solve(n_iter=100)
    bem = optimus.model.acoustics.Pmchwt(source, geometry, material_ext, material_int, "mass")
    bem.solve()
    iterations, converged = gmres_converged(bem)
    exact = np.nan_to_num(plane_field(analytical))
    numerical = np.nan_to_num(plane_field(bem))
    diff = numerical - exact
    results.append({
        "element_size": element_size,
        "vertices": int(geometry.number_of_vertices()),
        "gmres_iterations": int(iterations),
        "converged": bool(converged),
        "error_fro": float(np.linalg.norm(diff, "fro") / np.linalg.norm(exact, "fro")),
        "error_max": float(np.linalg.norm(diff, np.inf) / np.linalg.norm(exact, np.inf)),
        "time_s": round(time.time() - t0, 1),
        "fields": (exact, numerical),
    })
    r = results[-1]
    print("h = %.2f m: %4d vertices, %2d iterations, error %.4f (Frobenius) %.4f (max), %.1f s"
          % (element_size, r["vertices"], r["gmres_iterations"], r["error_fro"], r["error_max"], r["time_s"]))

## 3. Convergence plot

In [ ]:
h = np.array([r["element_size"] for r in results])
e_fro = np.array([r["error_fro"] for r in results])
e_max = np.array([r["error_max"] for r in results])
order = np.polyfit(np.log(h), np.log(e_fro), 1)[0]   # slope of log(error) against log(h)
print("Observed convergence order (Frobenius): %.2f" % order)

fig = plt.figure(figsize=(6, 5))
plt.loglog(h, e_fro, "go-", label="Frobenius norm")
plt.loglog(h, e_max, "bo-", label="maximum norm")
plt.loglog(h, e_fro[0] * (h / h[0]), "k:", label="slope 1")
plt.loglog(h, e_fro[0] * (h / h[0])**2, "k--", label="slope 2")
plt.gca().invert_xaxis()
plt.xlabel("element size [m]")
plt.ylabel("relative error")
plt.legend()
plt.grid(alpha=0.3, which="both")
save_figures(fig, NOTEBOOK_NAME)
plt.show()

RESULTS["errors"] = [{k: r[k] for k in ("element_size", "vertices", "gmres_iterations",
                                         "error_fro", "error_max", "time_s")} for r in results]
RESULTS["convergence_order"] = round(float(order), 2)

## 4. Coarsest and finest mesh side by side

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 10))
for row, r in enumerate((results[0], results[-1])):
    for col, (title, data) in enumerate((("Analytical", r["fields"][0]),
                                         ("OptimUS, h = %.2f m" % r["element_size"], r["fields"][1]))):
        im = axs[row][col].imshow(data, extent=bounding_box, vmin=-2, vmax=2, cmap="bwr")
        axs[row][col].set_title(title)
        axs[row][col].set_xlabel("x [m]")
        axs[row][col].set_ylabel("y [m]")
        fig.colorbar(im, ax=axs[row][col])
save_figures(fig, NOTEBOOK_NAME + "_fields")
plt.show()

## 5. Checks

Measured on 9 Nov 2026 in the Docker image. Full mode (250 x 250 grid): Frobenius error 0.106, 0.047,
0.027, 0.015 and 0.0081 for h = 1.0, 0.75, 0.5, 0.25 and 0.1 m, observed order 1.04. CI mode
(100 x 100 grid, h down to 0.25 m): 0.101, 0.049, 0.024, 0.014, order about 1.4.
The 1% limit for the 0.1 m mesh is success criterion N4 of the project (measured 0.81%); the 2% CI
limit is the measured 1.4% plus a margin. The order limit of 0.8 is below both measured orders but
still fails if the error stops falling with refinement.

In [ ]:
check("all_converged", all(r["converged"] for r in results),
      "GMRES iterations: " + str([r["gmres_iterations"] for r in results]))
check("error_decreases", bool(np.all(np.diff(e_fro) < 0)),
      "Frobenius errors " + str(np.round(e_fro, 4).tolist()))
check("finest_mesh_accurate", e_fro[-1] < finest_error_limit,
      "error %.4f at h = %.2f m, limit %.2f" % (e_fro[-1], h[-1], finest_error_limit))
check("convergence_order", order >= 0.8, "observed order %.2f, limit 0.8" % order)

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))